# Phase 1: Robust Baseline Model Engineering

This notebook is the first phase of the **Distributed Land Cover Classification Platform**. 
Our goal here is to train the "Brain" of our system: an **EfficientNetV2-Small** neural network.

To combat the **Domain Shift** problem (training on European data but deploying on Indian data), we strictly follow the Architecture Document and implement:
1. **Stratified K-Fold Cross-Validation** (K=5)
2. **CutMix** and **Mixup** Augmentations
3. **Focal Loss**

---

### Step 1: Install Dependencies
We install PyTorch and ONNX to export our trained weights. We will run inference natively in the browser using ONNX Runtime Web.

In [ ]:
!pip install onnx onnxruntime scikit-learn tqdm onnxscript

---
### Step 2: Mount Google Drive & Environment Setup
Since the EuroSAT dataset is in your Google Drive, we need to mount it first so Colab can read the file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import StratifiedKFold
import numpy as np
import os
import zipfile
import shutil
import copy

# Ensure we are using the Free Tier T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
### Step 3: Extract the Dataset from Google Drive
We will extract the `EuroSAT_RGB.zip` file from your Google Drive into the local Colab disk (`./data/`). This makes loading much faster than reading directly from Drive during training.

In [ ]:
# Update this if your zip file is in a folder (e.g. '/content/drive/MyDrive/Datasets/EuroSAT_RGB.zip')
zip_path = "/content/drive/MyDrive/EuroSAT_RGB.zip"
extract_path = "./data/"

if os.path.exists(zip_path):
    if not os.path.exists(extract_path + "EuroSAT_RGB") and not os.path.exists(extract_path + "2750"):
        print(f"Found {zip_path}, extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Extraction complete.")
    else:
        print("Dataset already extracted!")
else:
    print(f"ERROR: Could not find {zip_path}. Please check if the file is named exactly 'EuroSAT_RGB.zip' and if it is in the root of your Google Drive. If it's in a subfolder, change the zip_path variable.")

---
### Step 4: Dataset Loading & Taxonomy Mapping
EuroSAT originally comes with 10 classes. Our application requires a simplified **6-class taxonomy**.
We enforce this mapping directly at the Dataset level so the model only ever learns 6 output logits. We also apply standard ImageNet normalization and resize images to 224x224.

In [ ]:
class CustomEuroSAT(Dataset):
    def __init__(self, root, transform=None):
        self.dataset = ImageFolder(root=root)
        self.transform = transform
        
        # Mapping from ImageFolder class indices to our custom taxonomy.
        self.mapping = {
            0: 0, 6: 0, 5: 0, # Crop
            1: 1,             # Forest
            2: 2,             # HerbaceousVegetation
            3: 3,             # Highway
            4: 4, 7: 4,       # Urban
            8: 5, 9: 5        # WaterBodies
        }
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        mapped_label = self.mapping[label]
        if self.transform:
            img = self.transform(img)
        return img, mapped_label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Find where the classes are actually extracted
data_dir = "./data/EuroSAT_RGB"
if not os.path.exists(data_dir):
    if os.path.exists("./data/2750"):
        data_dir = "./data/2750"
    else:
        data_dir = "./data/"

print(f"Loading dataset from {data_dir}...")
full_dataset = CustomEuroSAT(root=data_dir, transform=transform)
print(f"Dataset size: {len(full_dataset)}")

---
### Step 5: Advanced Loss and Augmentations (Domain Adaptation)
Here we implement the core strategies to resist Domain Shift:
- **Focal Loss**: Focuses heavily on hard/ambiguous boundaries.
- **Mixup**: Blends two images linearly.
- **CutMix**: Cuts a patch from one image and pastes it into another, forcing the network to learn structural context rather than just pixel colors.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def augmented_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

---
### Step 6: Model Definition (EfficientNetV2-Small)
We use `EfficientNetV2-S` because it achieves State-of-the-Art accuracy while being incredibly lightweight (~24M parameters), which is crucial for compiling and running smoothly in a WebGL browser environment later.

In [ ]:
def get_model():
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
    # Overwrite the final classification layer for our 6 classes
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, 6)
    return model.to(device)

---
### Step 7: ROBUST Stratified K-Fold Training Loop (With Checkpointing)
This loop will now save a **Checkpoint** directly to your Google Drive at the end of *every single epoch*. 
If Colab disconnects, you can simply run this cell again. It will automatically load the checkpoint from your Google Drive and resume training exactly where it left off, so you **never lose progress!**

In [ ]:
epochs_per_fold = 5
k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
from tqdm import tqdm

print("Extracting labels for Stratified K-Fold splitting...")
targets = [full_dataset[i][1] for i in range(len(full_dataset))]

# =============== CHECKPOINTING LOGIC ===============
checkpoint_path = "/content/drive/MyDrive/eurosat_checkpoint.pth"
start_fold = 0
start_epoch = 0
best_overall_acc = 0.0
best_model_wts = None

if os.path.exists(checkpoint_path):
    print(f"\n🟢 Found existing checkpoint at {checkpoint_path}!")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    start_fold = checkpoint['fold']
    start_epoch = checkpoint['epoch'] + 1
    best_overall_acc = checkpoint['best_overall_acc']
    best_model_wts = checkpoint['best_model_wts']
    print(f"Resuming training from Fold {start_fold + 1}, Epoch {start_epoch + 1}. Current Best Acc: {best_overall_acc:.2f}%")
else:
    print("\n🔴 No checkpoint found in Google Drive. Starting training from scratch.")
# ===================================================

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
    if fold < start_fold:
        continue # Skip folds we've already completed before the disconnect
        
    print(f"\n{'='*20} Fold {fold+1}/{k_folds} {'='*20}")
    
    train_sub = torch.utils.data.Subset(full_dataset, train_idx)
    val_sub = torch.utils.data.Subset(full_dataset, val_idx)
    
    train_loader = DataLoader(train_sub, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_sub, batch_size=32, shuffle=False, num_workers=2)
    
    model = get_model() # Get a fresh model for each fold
    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    
    # If resuming mid-fold, load the model and optimizer states
    if fold == start_fold and os.path.exists(checkpoint_path):
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
    # Determine where to start the epoch loop
    current_start_epoch = start_epoch if fold == start_fold else 0
    
    for epoch in range(current_start_epoch, epochs_per_fold):
        model.train()
        running_loss = 0.0
        
        # Wrap train_loader with tqdm for a real-time progress bar
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_per_fold}")
        
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # 50% chance Mixup, 50% chance CutMix
            if np.random.rand() > 0.5:
                inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=0.2)
            else:
                inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha=1.0)
                
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = augmented_criterion(criterion, outputs, targets_a, targets_b, lam)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
            # Update the progress bar with the rolling loss
            pbar.set_postfix({'loss': f"{running_loss / (pbar.n + 1):.4f}"})
            
        # Validation phase
        model.eval()
        correct = 0; total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        val_acc = 100 * correct / total
        print(f"Val Acc: {val_acc:.2f}%")
        
        # Check if this is the absolute best model across all folds and epochs
        if val_acc > best_overall_acc:
            best_overall_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            
        # =============== SAVE CHECKPOINT TO GOOGLE DRIVE ===============
        torch.save({
            'fold': fold,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_overall_acc': best_overall_acc,
            'best_model_wts': best_model_wts
        }, checkpoint_path)
        print(f"💾 Progress saved permanently to Google Drive ({checkpoint_path})")

print(f"\n🏆 Training Complete! Best Overall Accuracy: {best_overall_acc:.2f}%")

# Load the absolute best weights into our final model to export
final_model = get_model()
final_model.load_state_dict(best_model_wts)

---
### Step 8: Edge Export Pipeline (ONNX)
We export the absolute best PyTorch model out of all 5 folds natively to `.onnx`. We do NOT need to convert it to TensorFlow.js, because we will use `onnxruntime-web` in our frontend which is much faster and supports WebGPU natively!

In [ ]:
final_model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = "efficientnet_v2_s.onnx"

torch.onnx.export(final_model, dummy_input, onnx_path, 
                  export_params=True, 
                  opset_version=18, 
                  do_constant_folding=True, 
                  input_names=['input'], 
                  output_names=['output'])

print("✅ Exported to ONNX successfully!")
print(f"Please open the file browser on the left in Colab (folder icon) and download '{onnx_path}' directly to your computer.")